In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read data as raw lines (single column)
df_benchmark = pd.read_csv("S:\\MasterThesisData\\BenchmarkResults\\SCALE_ADAPTIVE\\benchmark_breast_ultrasound_malignant.csv", header=None, names=["raw"])

# Parse semicolon-separated data:
# [File, Threshold, Beta, DiceCoefficient, Hist_0 ... Hist_100]
split_data = df_benchmark["raw"].astype(str).str.rstrip(";").str.split(";", expand=True)

hist_cols = [f"Hist_{i}" for i in range(101)]

# Ignore first column (file path) for now
parsed = split_data.iloc[:, 1:1 + 3 + len(hist_cols)].copy()
parsed.columns = ["Threshold", "Beta", "DiceCoefficient", *hist_cols]

parsed = parsed.apply(pd.to_numeric, errors="coerce")

# Filter out rows where DiceCoefficient is zero
parsed = parsed[parsed["DiceCoefficient"] > 0]

# Print total number of valid rows
print(f"Total valid rows: {len(parsed)}")

# Average out the DiceCoefficient for each combination of Threshold and Beta
parsed = parsed.groupby(["Threshold", "Beta"], as_index=False)["DiceCoefficient"].mean()

# Build grid for surface plot
grid = parsed.pivot(index="Beta", columns="Threshold", values="DiceCoefficient")
x = grid.columns.to_numpy()
y = grid.index.to_numpy()
X, Y = np.meshgrid(x, y)
Z = grid.to_numpy()

# 2.5D surface plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")

ax.set_xlabel("Threshold")
ax.set_ylabel("Beta")
ax.set_zlabel("DiceCoefficient")
ax.view_init(elev=30, azim=225)
#ax.set_title("DiceCoefficient vs Threshold and Beta")
fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1, label="DiceCoefficient")

plt.tight_layout()
plt.show()

In [ ]:
# Average DiceCoefficient over compactness for each superpixel count
mean_by_threshold = grid.mean(axis=0)

# Average DiceCoefficient over superpixel counts for each compactness value
mean_by_beta = grid.mean(axis=1)

fig_avg, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mean_by_threshold.index, mean_by_threshold.values, marker="o", linewidth=1.8)
axes[0].set_title("Avg DiceCoefficient vs Threshold")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Avg DiceCoefficient")
axes[0].grid(alpha=0.3)

axes[1].plot(mean_by_beta.index, mean_by_beta.values, marker="o", linewidth=1.8)
axes[1].set_title("Avg DiceCoefficient vs Beta")
axes[1].set_xlabel("Beta")
axes[1].set_ylabel("Avg DiceCoefficient")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Parallel coordinates view of the same DiceCoefficient surface data
pc_df = grid.reset_index().copy()
pc_df["Compactness"] = pc_df["Compactness"].astype(int).astype(str)

ax_pc = pd.plotting.parallel_coordinates(
	pc_df,
	class_column="Compactness",
	colormap="viridis",
	linewidth=1.8,
	alpha=0.9
)

ax_pc.set_title("Dice coeff for multiple compactness and SP counts")
ax_pc.set_xlabel("Superpixels")
ax_pc.set_ylabel("DiceCoefficient")
plt.xticks(rotation=45)
plt.legend(title="Compactness", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Re-parse full benchmark rows to recover histogram bins
full = df_benchmark.iloc[:, 0].str.rstrip(";").str.split(";", expand=True)
full_cols = ["Superpixels", "Compactness", "DiceCoefficient", *hist_cols]
full = full.iloc[:, :len(full_cols)]
full.columns = full_cols
full = full.apply(pd.to_numeric, errors="coerce")

# Filter
full = full[full["DiceCoefficient"] > 0]

# Average histogram value per bin across all rows
hist_mean = full[hist_cols].mean(axis=0)
bin_idx = np.arange(len(hist_cols))



plt.figure(figsize=(10, 4))
plt.plot(bin_idx, hist_mean.values, marker="o", linewidth=1.5, markersize=3)
plt.fill_between(bin_idx, hist_mean.values, alpha=0.2)
plt.title("Distribution of SP mask overlap with GT mask")
plt.xlabel("Percentage of SP mask in GT mask")
plt.ylabel("SP count")
plt.yscale("log")
plt.xticks(np.arange(0, len(hist_cols), 10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
compactness_values = sorted(full["Compactness"].dropna().unique())
bin_idx = np.arange(len(hist_cols))

surface_df = full.groupby("Compactness")[hist_cols].mean().reindex(compactness_values)

X, Y = np.meshgrid(bin_idx, surface_df.index.to_numpy())
Z = surface_df.to_numpy()

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(
	X,
	Y,
	np.log10(Z + 1),
	cmap="viridis",
	edgecolor="none"
)

ax.set_title("Distribution of SP mask overlap with GT mask by compactness")
ax.set_xlabel("Percentage of SP mask in GT mask")
ax.set_ylabel("Compactness")
ax.set_zlabel("log10(SP count + 1)")
ax.set_zlim(0,0.5)
ax.set_xticks(np.arange(0, len(hist_cols), 20))
ax.view_init(elev=15, azim=300)

fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1, label="log10(SP count + 1)")
plt.tight_layout()
plt.show()